In [0]:
%run ../../src/utils/config

In [0]:
%python
import os

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

config = load_config(config_path)

In [0]:
%python
catalog = config["target"]["catalog"]
gold_schema = config["target"]["gold_schema"]

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

In [0]:
%python
#============================================================
#ddl_gold_dim_pokemon
#============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_pokemon (
    pokemon_key       BIGINT GENERATED ALWAYS AS IDENTITY,
    pokemon_id        INT NOT NULL,
    name              STRING,
    base_experience   INT,
    height            INT,
    weight            INT,
    is_default        BOOLEAN,
    pokemon_order     INT,
    species_name      STRING,
    species_url       STRING,

    effective_from    TIMESTAMP NOT NULL,
    effective_to      TIMESTAMP,
    is_current        BOOLEAN NOT NULL,
    attribute_hash    STRING,

    gold_created_at   TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_type
#============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_type (
    type_key        BIGINT GENERATED ALWAYS AS IDENTITY,
    type_url        STRING NOT NULL,
    type_name       STRING NOT NULL,
    gold_created_at TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_ability
#============================================================   
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_ability (
    ability_key      BIGINT GENERATED ALWAYS AS IDENTITY,
    ability_url      STRING NOT NULL,
    ability_name     STRING NOT NULL,
    gold_created_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_move
#============================================================  
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_move (
    move_key         BIGINT GENERATED ALWAYS AS IDENTITY,
    move_url         STRING NOT NULL,
    move_name        STRING NOT NULL,
    gold_created_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_form
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_form (
    form_key         BIGINT GENERATED ALWAYS AS IDENTITY,
    form_url         STRING NOT NULL,
    form_name        STRING NOT NULL,
    gold_created_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_version_group
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_version_group (
    version_group_key  BIGINT GENERATED ALWAYS AS IDENTITY,
    version_group_url  STRING NOT NULL,
    version_group_name STRING NOT NULL,
    gold_created_at    TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_move_learning_method
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_learn_method (
    learn_method_key  BIGINT GENERATED ALWAYS AS IDENTITY,
    learn_method_url  STRING NOT NULL,
    learn_method_name STRING NOT NULL,
    gold_created_at   TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_game_version
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_game_version (
    game_version_key BIGINT GENERATED ALWAYS AS IDENTITY,
    version_url      STRING NOT NULL,
    version_name     STRING NOT NULL,
    gold_created_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_dim_stat
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.dim_stat (
    stat_key         BIGINT GENERATED ALWAYS AS IDENTITY,
    stat_url         STRING NOT NULL,
    stat_name        STRING NOT NULL,
    gold_created_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_bridge_pokemon_to_type
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.bridge_pokemon_type (
    pokemon_key     BIGINT NOT NULL,
    type_key        BIGINT NOT NULL,
    type_slot       INT NOT NULL,
    gold_loaded_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
%python
#============================================================
#ddl_gold_bridge_pokemon_to_ability
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.bridge_pokemon_ability (
    pokemon_key     BIGINT NOT NULL,
    ability_key     BIGINT NOT NULL,
    ability_slot    INT NOT NULL,
    is_hidden       BOOLEAN,
    gold_loaded_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
#============================================================
#ddl_gold_bridge_pokemon_to_form
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.bridge_pokemon_form (
    pokemon_key     BIGINT NOT NULL,
    form_key        BIGINT NOT NULL,
    gold_loaded_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
#============================================================
#ddl_gold_fact_pokemon_stat
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.fact_pokemon_stats (
    pokemon_key     BIGINT NOT NULL,
    stat_key        BIGINT NOT NULL,
    base_stat       INT,
    effort          INT,
    gold_loaded_at  TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
#============================================================
#ddl_gold_fact_pokemon_move_learning
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.fact_pokemon_move_learning  (
    pokemon_key        BIGINT NOT NULL,
    move_key           BIGINT NOT NULL,
    version_group_key  BIGINT NOT NULL,
    learn_method_key   BIGINT NOT NULL,
    level_learned_at   INT,
    learn_order        INT,
    gold_loaded_at     TIMESTAMP NOT NULL
)
USING DELTA
""")

In [0]:
#============================================================
#ddl_gold_fact_pokemon_game_presence
#============================================================ 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{gold_schema}.fact_pokemon_game_presence  (
    pokemon_key       BIGINT NOT NULL,
    game_version_key  BIGINT NOT NULL,
    game_index        INT,
    gold_loaded_at    TIMESTAMP NOT NULL
)
USING DELTA
""")